##### `Semantic Search with Qdrant and embeddings using HuggingFace Model`

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from tqdm import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import (
    VectorParams, Distance, Batch, Filter, 
    MatchValue, FieldCondition, PayloadSchemaType
)

d:\AAAAAAA\RAG\Projects\Vector_Database_Qdrant-Semantic\Vector_Database_Qdrant-Semantic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# Load dotenv file
load_dotenv(override=True)
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")


In [3]:

# Read the CSV File
FILE_PATH = os.path.join(os.getcwd(), 'data', 'articles_new.csv')
try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    # Create sample data if file not found
    data = {
        'id': list(range(500)),
        'title': [f"Sample Article Title {i}" for i in range(500)]
    }
    df = pd.DataFrame(data)

# Add class column and limit to 500 rows
df['class'] = ['class-a', 'class-b'] * (len(df) // 2 + (1 if len(df) % 2 else 0))
df = df.iloc[:500]
print(f"Dataset loaded with {len(df)} articles")


Dataset loaded with 500 articles


In [4]:
# Initialize embedding models
print("Loading embedding models...")
model_hugging_6 = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', device='cpu')
model_hugging_12 = SentenceTransformer(model_name_or_path='all-MiniLM-L12-v2', device='cpu')

# Test embedding dimensions
vect_length_hugging_6 = len(model_hugging_6.encode(df['title'].iloc[0]))
vect_length_hugging_12 = len(model_hugging_12.encode(df['title'].iloc[0]))
print(f'Embedding dimension for all-MiniLM-L6-v2: {vect_length_hugging_6}')
print(f'Embedding dimension for all-MiniLM-L12-v2: {vect_length_hugging_12}')

# Connect to Qdrant
print("Connecting to Qdrant...")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60.0)  # Increased timeout

# Collection names
collec_MiniLM_L6 = 'all-MiniLM-L6-v2'
collec_MiniLM_L12 = 'all-MiniLM-L12-v2'


Loading embedding models...
Embedding dimension for all-MiniLM-L6-v2: 384
Embedding dimension for all-MiniLM-L12-v2: 384
Connecting to Qdrant...


In [5]:
# Create collections properly
print("Creating collections with payload indexes...")
for collection_name, vector_size in [(collec_MiniLM_L6, vect_length_hugging_6), (collec_MiniLM_L12, vect_length_hugging_12)]:
    # Check if collection exists
    try:
        client.get_collection(collection_name)
        print(f"Collection {collection_name} already exists, deleting it...")
        client.delete_collection(collection_name)
    except Exception:
        pass  # Collection doesn't exist, which is fine
    
    # Create collection with proper configuration
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE,
            on_disk=True
        )
    )
    
    # Create payload index for 'class' field
    try:
        client.create_payload_index(
            collection_name=collection_name,
            field_name="class",
            field_schema=PayloadSchemaType.KEYWORD
        )
        print(f'Collection {collection_name} created with class index')
    except Exception as e:
        print(f"Error creating index: {e}")
        print("Continuing anyway...")


Creating collections with payload indexes...
Collection all-MiniLM-L6-v2 already exists, deleting it...
Collection all-MiniLM-L6-v2 created with class index
Collection all-MiniLM-L12-v2 already exists, deleting it...
Collection all-MiniLM-L12-v2 created with class index


In [6]:
# Upload data to collections with smaller batch size and retry logic
print("Uploading data to collections...")

def upsert_with_retry(collection_name, model, batch_size=16, max_retries=3):
    print(f"Uploading to {collection_name}...")
    failed_batches = []
    
    for batch_start in tqdm(range(0, len(df), batch_size)):
        batch_end = min(batch_start + batch_size, len(df))
        batch_df = df.iloc[batch_start:batch_end]
        
        # Extract batch components
        titles = batch_df['title'].tolist()
        ids = batch_df['id'].tolist()
        classes = batch_df['class'].tolist()
        
        # Create embeddings
        embeddings = model.encode(titles).tolist()
        
        # Create payloads
        payloads = [{'class': cls} for cls in classes]
        
        # Prepare batch
        batch = Batch(ids=ids, vectors=embeddings, payloads=payloads)
        
        # Try to upload with retries
        success = False
        for attempt in range(max_retries):
            try:
                client.upsert(collection_name=collection_name, points=batch, wait=True)
                success = True
                break
            except Exception as e:
                print(f"Error in batch {batch_start}-{batch_end} (attempt {attempt+1}/{max_retries}): {e}")
                time.sleep(2)  # Wait before retrying
        
        if not success:
            failed_batches.append((batch_start, batch_end))
    
    return failed_batches


Uploading data to collections...


In [7]:
# Upload with smaller batch size and retries
failed_L6 = upsert_with_retry(collec_MiniLM_L6, model_hugging_6, batch_size=16)
failed_L12 = upsert_with_retry(collec_MiniLM_L12, model_hugging_12, batch_size=16)

# Report any failures
if failed_L6:
    print(f"Failed to upload {len(failed_L6)} batches to {collec_MiniLM_L6}")
if failed_L12:
    print(f"Failed to upload {len(failed_L12)} batches to {collec_MiniLM_L12}")

# Check collection status
for collection in [collec_MiniLM_L6, collec_MiniLM_L12]:
    try:
        collection_info = client.get_collection(collection_name=collection)
        print(f'{collection}: Status is: {collection_info.status}')
        print(f'{collection}: Vectors Count is: {collection_info.points_count}')
    except Exception as e:
        print(f"Error checking collection {collection}: {e}")

# Run a query with class filter
print("\nRunning filtered query...")
query_text = 'Neutral Technology'

try:
    # Generate embedding for query
    query_embedding = model_hugging_12.encode(query_text).tolist()
    
    # Run filtered search using query_points
    try:
        results = client.query_points(
            collection_name=collec_MiniLM_L12,
            query_vector=query_embedding,
            limit=10,
            query_filter=Filter(must=[FieldCondition(key='class', match=MatchValue(value='class-a'))])
        )
        
        # Display filtered results
        print("\nFiltered Query Results (class-a only):")
        for i, point in enumerate(results):
            print(f"{i+1}. ID: {point.id}, Score: {point.score:.4f}, Class: {point.payload['class']}")
    except Exception as e:
        # If query_points fails, try the deprecated search method
        print(f"Error using query_points, trying search instead: {e}")
        results = client.search(
            collection_name=collec_MiniLM_L12,
            query_vector=query_embedding,
            limit=10,
            query_filter=Filter(must=[FieldCondition(key='class', match=MatchValue(value='class-a'))])
        )
        
        # Display filtered results
        print("\nFiltered Query Results (class-a only):")
        for i, point in enumerate(results):
            print(f"{i+1}. ID: {point.id}, Score: {point.score:.4f}, Class: {point.payload['class']}")
    
    # Run unfiltered query for comparison
    try:
        unfiltered_results = client.query_points(
            collection_name=collec_MiniLM_L12,
            query_vector=query_embedding,
            limit=10
        )
    except Exception:
        # Fall back to deprecated method if query_points isn't available
        unfiltered_results = client.search(
            collection_name=collec_MiniLM_L12,
            query_vector=query_embedding,
            limit=10
        )
    
    # Display unfiltered results
    print("\nUnfiltered Query Results:")
    for i, point in enumerate(unfiltered_results):
        print(f"{i+1}. ID: {point.id}, Score: {point.score:.4f}, Class: {point.payload['class']}")
        
except Exception as e:
    print(f"Error during query: {e}")

print("\nProcess completed!")

Uploading to all-MiniLM-L6-v2...


100%|██████████| 32/32 [00:18<00:00,  1.72it/s]


Uploading to all-MiniLM-L12-v2...


100%|██████████| 32/32 [00:37<00:00,  1.18s/it]


all-MiniLM-L6-v2: Status is: green
all-MiniLM-L6-v2: Vectors Count is: 500
all-MiniLM-L12-v2: Status is: green
all-MiniLM-L12-v2: Vectors Count is: 500

Running filtered query...
Error using query_points, trying search instead: Unknown arguments: ['query_vector']


C:\Users\user\AppData\Local\Temp\ipykernel_27248\2562917051.py:44: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(



Filtered Query Results (class-a only):
1. ID: 3552, Score: 0.8089, Class: class-a
2. ID: 3368, Score: 0.3189, Class: class-a
3. ID: 3084, Score: 0.3146, Class: class-a
4. ID: 3486, Score: 0.2478, Class: class-a
5. ID: 3246, Score: 0.2375, Class: class-a
6. ID: 3124, Score: 0.2250, Class: class-a
7. ID: 3506, Score: 0.2129, Class: class-a
8. ID: 3150, Score: 0.2089, Class: class-a
9. ID: 3422, Score: 0.2064, Class: class-a
10. ID: 3174, Score: 0.1990, Class: class-a


C:\Users\user\AppData\Local\Temp\ipykernel_27248\2562917051.py:65: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  unfiltered_results = client.search(



Unfiltered Query Results:
1. ID: 3552, Score: 0.8089, Class: class-a
2. ID: 3368, Score: 0.3189, Class: class-a
3. ID: 3107, Score: 0.3169, Class: class-b
4. ID: 3084, Score: 0.3146, Class: class-a
5. ID: 3393, Score: 0.2806, Class: class-b
6. ID: 3415, Score: 0.2712, Class: class-b
7. ID: 3429, Score: 0.2479, Class: class-b
8. ID: 3486, Score: 0.2478, Class: class-a
9. ID: 3246, Score: 0.2375, Class: class-a
10. ID: 3277, Score: 0.2318, Class: class-b

Process completed!


# END